# Exploración Inicial — International Football Results (1872–2025)

Primer contacto con los datos. El objetivo es entender la estructura, 
identificar problemas de calidad y tomar decisiones de limpieza antes 
del análisis exploratorio.

**Dataset:** [martj42 en Kaggle](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)  
**Descargado:** junio 2026

In [2]:
import pandas as pd
import numpy as np

# Cargamos los 4 archivos
results      = pd.read_csv('../data/raw/results.csv')
goalscorers  = pd.read_csv('../data/raw/goalscorers.csv')
shootouts    = pd.read_csv('../data/raw/shootouts.csv')
former_names = pd.read_csv('../data/raw/former_names.csv')

print("✅ Archivos cargados correctamente")
print(f"   results:      {results.shape}")
print(f"   goalscorers:  {goalscorers.shape}")
print(f"   shootouts:    {shootouts.shape}")
print(f"   former_names: {former_names.shape}")

✅ Archivos cargados correctamente
   results:      (49493, 9)
   goalscorers:  (47821, 8)
   shootouts:    (678, 5)
   former_names: (36, 4)


## 1. Carga de datos

Cuatro archivos. El dataset tiene más estructura de la esperada: 
`former_names.csv` mapea nombres históricos de países a sus nombres 
actuales, útil si se cruza con fuentes externas.

In [3]:
# ¿Cómo luce results?
print("=== RESULTS ===")
print(results.head())
print()

# Tipos de datos y valores nulos
print("=== TIPOS Y NULOS ===")
print(results.info())

=== RESULTS ===
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  

=== TIPOS Y NULOS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49493 entries, 0 to 49492
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49493 non-null  object 
 1   home_team   49493 non-null  object 
 2   away_team   49493 non-null  object 
 3   home_score  49477 

### Observaciones

- `date` viene como `object` (texto) — hay que convertirlo a datetime
- `home_score` y `away_score` son `float64` siendo conteos de goles. 
  Pasa porque hay 16 valores NaN: pandas no puede mezclar NaN con int
- Sin nulos en columnas clave (`home_team`, `away_team`, `tournament`)
- `former_names`: el autor ya unificó nombres en `results.csv`. 
  "Zaïre" no aparece — aparece "DR Congo". No hay trabajo de 
  normalización pendiente en este dataset.

In [4]:
print("=== FORMER_NAMES ===")
print(former_names.head(10))

=== FORMER_NAMES ===
          current                                former  start_date  \
0           Benin                               Dahomey  1959-11-08   
1    Burkina Faso                           Upper Volta  1960-04-14   
2         Curaçao                  Netherlands Antilles  1957-03-03   
3  Czechoslovakia                               Bohemia  1903-04-05   
4  Czechoslovakia                   Bohemia and Moravia  1939-01-01   
5  Czechoslovakia  Representation of Czechs and Slovaks  1993-03-24   
6        DR Congo                         Belgian Congo  1948-05-25   
7        DR Congo                    Congo-Léopoldville  1963-04-12   
8        DR Congo                        Congo-Kinshasa  1965-01-09   
9        DR Congo                                 Zaïre  1971-01-10   

     end_date  
0  1975-11-30  
1  1984-08-04  
2  2010-10-10  
3  1919-01-01  
4  1945-05-01  
5  1993-11-17  
6  1956-01-02  
7  1964-07-19  
8  1970-11-24  
9  1997-04-27  


In [5]:
# ¿Quiénes son los 16 partidos sin resultado?
sin_resultado = results[results['home_score'].isna()]
print(f"Partidos sin resultado: {len(sin_resultado)}")
print()
print(sin_resultado[['date', 'home_team', 'away_team', 'tournament']].to_string())

Partidos sin resultado: 16

             date      home_team               away_team      tournament
49477  2026-06-28   South Africa                  Canada  FIFA World Cup
49478  2026-06-29         Brazil                   Japan  FIFA World Cup
49479  2026-06-29        Germany                Paraguay  FIFA World Cup
49480  2026-06-29    Netherlands                 Morocco  FIFA World Cup
49481  2026-06-30    Ivory Coast                  Norway  FIFA World Cup
49482  2026-06-30         France                  Sweden  FIFA World Cup
49483  2026-06-30         Mexico                 Ecuador  FIFA World Cup
49484  2026-07-01        England                DR Congo  FIFA World Cup
49485  2026-07-01        Belgium                 Senegal  FIFA World Cup
49486  2026-07-01  United States  Bosnia and Herzegovina  FIFA World Cup
49487  2026-07-02          Spain                 Austria  FIFA World Cup
49488  2026-07-02       Portugal                 Croatia  FIFA World Cup
49489  2026-07-02    Sw

### Partidos sin resultado: fixture futuro

Los 16 registros sin score no son datos faltantes — son partidos del 
Mundial 2026 ya cargados como fixture (algunos se juegan esta semana). 
Se descartan del análisis por no tener resultado real.

**Nota de reproducibilidad:** quien descargue este dataset en el 
futuro tendrá esos partidos con resultado. Este análisis cubre datos 
hasta junio de 2026.

In [6]:
# Panorama general
print(f"Rango de fechas:    {results['date'].min()} → {results['date'].max()}")
print(f"Selecciones únicas: {pd.unique(results[['home_team', 'away_team']].values.ravel()).shape[0]}")
print(f"Torneos únicos:     {results['tournament'].nunique()}")
print()
print("Top 10 torneos por número de partidos:")
print(results['tournament'].value_counts().head(10))

Rango de fechas:    1872-11-30 → 2026-07-03
Selecciones únicas: 336
Torneos únicos:     200

Top 10 torneos por número de partidos:
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1052
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64


In [7]:
# Diagnóstico completo antes de limpiar
import datetime

hoy = pd.Timestamp('today').normalize()

# Partidos futuros (sin resultado)
futuros = results[results['home_score'].isna()]

# Partidos históricos (con resultado)
historicos = results[results['home_score'].notna()]

print(f"Partidos con resultado:    {len(historicos):,}")
print(f"Partidos sin resultado:    {len(futuros):,}  ← fixture futuro, los descartamos")
print()

# ¿Cuántos equipos tienen nombres históricos?
# (equipos que aparecen en results pero ya no existen con ese nombre)
equipos_en_results = set(pd.unique(results[['home_team', 'away_team']].values.ravel()))
equipos_renombrados = set(former_names['former'].unique())
equipos_con_nombre_viejo = equipos_en_results & equipos_renombrados

print(f"Selecciones únicas en results:          {len(equipos_en_results)}")
print(f"Nombres históricos en former_names:     {len(equipos_renombrados)}")
print(f"Nombres viejos que aparecen en results: {len(equipos_con_nombre_viejo)}")
print()
if equipos_con_nombre_viejo:
    print("Ejemplos:", list(equipos_con_nombre_viejo)[:8])

Partidos con resultado:    49,477
Partidos sin resultado:    16  ← fixture futuro, los descartamos

Selecciones únicas en results:          336
Nombres históricos en former_names:     36
Nombres viejos que aparecen en results: 0



In [11]:
total = len(historicos)
por_tipo = historicos.copy()
por_tipo['es_amistoso'] = por_tipo['tournament'] == 'Friendly'

print("Distribución Amistoso vs Oficial:")
print(por_tipo['es_amistoso'].value_counts().rename({True: 'Amistoso', False: 'Oficial'}))
print()
print(f"Amistosos: {por_tipo['es_amistoso'].sum():,} ({por_tipo['es_amistoso'].mean()*100:.1f}%)")
print(f"Oficiales: {(~por_tipo['es_amistoso']).sum():,} ({(~por_tipo['es_amistoso']).mean()*100:.1f}%)")

Distribución Amistoso vs Oficial:
es_amistoso
Oficial     31089
Amistoso    18388
Name: count, dtype: int64

Amistosos: 18,388 (37.2%)
Oficiales: 31,089 (62.8%)


## 2. Decisiones de limpieza

Con base en el diagnóstico anterior:

1. Eliminar los 16 partidos sin resultado (fixture futuro)
2. Convertir `date` a datetime
3. Convertir goles a entero (sin NaN ya no hay obstáculo)
4. Agregar columnas `year` y `decade` para análisis temporal
5. Crear columna `result` como variable objetivo del modelo

**Sobre amistosos (37.2% del dataset):** se incluyen. 
Se creará una feature `is_friendly` para que el modelo aprenda 
empíricamente cuánto peso darles, en lugar de descartarlos a mano.

In [9]:
# ============================================================
# LIMPIEZA BASE DE results.csv
# ============================================================

df = results.copy()  # Nunca tocamos el original

# 1. Eliminar partidos sin resultado (fixture futuro)
df = df[df['home_score'].notna()].copy()
print(f"Tras eliminar futuros: {len(df):,} partidos")

# 2. Convertir fecha a datetime
df['date'] = pd.to_datetime(df['date'])

# 3. Convertir goles a entero (ya no hay NaN)
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

# 4. Extraer año y década (útiles para análisis temporal)
df['year']   = df['date'].dt.year
df['decade'] = (df['year'] // 10) * 10

# 5. Crear resultado desde perspectiva del local
def resultado_local(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

df['result'] = df.apply(resultado_local, axis=1)

# Verificación final
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nDistribución de resultados:")
print(df['result'].value_counts())
print(f"\nMuestra:")
print(df.head(3))

Tras eliminar futuros: 49,477 partidos

Tipos de datos:
date          datetime64[ns]
home_team             object
away_team             object
home_score             int64
away_score             int64
tournament            object
city                  object
country               object
neutral                 bool
year                   int32
decade                 int32
result                object
dtype: object

Distribución de resultados:
result
home_win    24245
away_win    13979
draw        11253
Name: count, dtype: int64

Muestra:
        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   

    country  neutral  year  decade    result  
0  Scotland    False  1872    1870      draw  
1   England    False  1873    1870  home_

In [10]:
# Guardar dataset limpio
df.to_csv('../data/processed/results_clean.csv', index=False)
print(f"✅ Guardado: data/processed/results_clean.csv")
print(f"   {len(df):,} partidos × {df.shape[1]} columnas")

✅ Guardado: data/processed/results_clean.csv
   49,477 partidos × 12 columnas


## 3. Conclusiones de esta exploración

- Dataset limpio: **49,477 partidos** × 12 columnas
- La distribución de resultados ya revela un hallazgo: el local gana 
  el **49%** de los partidos en 153 años de historia
- El dataset está desbalanceado (`home_win` duplica a `draw`): 
  accuracy no será la métrica principal del modelo
- Se explora en detalle en `02_analisis_exploratorio.ipynb`